# Run MODFLOW-USG with FloPy

Local FloPy workflow (no Tapis calls):
1. Load an existing model from shared model storage
2. Point it to a local tutorial output workspace
3. Write and run model input
4. Check expected output files

This notebook is a compact local FloPy workflow for running an existing MODFLOW-USG model. It is intentionally shorter than the Gulf model notebook, but the same pattern applies: configure paths, prepare the model workspace, run the model, and verify outputs before interpretation.

## Before You Run

- Make sure the model input files are available from the configured shared storage path.
- Make sure the correct MODFLOW executable is available for this model version.
- Review the path variables before running cells that stage files or write outputs.

## Expected Outputs

- A local tutorial run workspace is created under `model_output_directory/`.
- The existing model is loaded or staged for inspection with FloPy.
- The model run reports whether it succeeded.
- Expected model output files are checked before moving on.


## Imports And Path Setup

This cell imports FloPy and supporting utilities, then sets the local paths used by the run. Review the model source, output directory, and executable settings before continuing.


In [ ]:
from pathlib import Path

import flopy


## Use Shared Model Inputs

The notebook uses model inputs and MODFLOW executables that are already staged on the shared TACC filesystem, then prepares a local tutorial workspace for generated output.


In [ ]:
# Paths and executable configuration
modeldir = Path(r"/corral-repl/tacc/aci/PT2050/projects/PTDATAX-272/workingGAMs/Carrizo-Wilcox-central/gmv-modflow-usg-Modified")
run_dir = Path(r"model_output_directory/modflow_usg")
exe_name = r"/corral-repl/tacc/aci/PT2050/community/DSO-Institute-2026/flopy/bin/mfusg"

print(f"FloPy version: {flopy.__version__}")
print(f"Source model directory: {modeldir}")
print(f"Run workspace: {run_dir}")
print(f"Executable: {exe_name}")

if not modeldir.exists():
    raise FileNotFoundError(f"Model directory not found: {modeldir}")

exe_path = Path(exe_name)
if not exe_path.is_file():
    raise FileNotFoundError(f"MODFLOW executable not found: {exe_path}")

print(f"Found executable: {exe_path}")
run_dir.mkdir(parents=True, exist_ok=True)


## Load The Existing Model

FloPy loads the existing MODFLOW-USG model so students can inspect how the unstructured-grid model is represented in Python.


In [ ]:
preferred_namefiles = ["gma12.mod.nam", "gma12.nam", "model.nam"]
namefile = next((n for n in preferred_namefiles if (modeldir / n).exists()), None)
if namefile is None:
    nam_candidates = sorted(modeldir.glob("*.nam"))
    if not nam_candidates:
        raise FileNotFoundError(f"No MODFLOW-USG .nam file found in {modeldir}")
    namefile = nam_candidates[0].name

print(f"Using name file: {namefile}")

model_obj = flopy.mfusg.MfUsg.load(
    f=namefile,
    version="mfusg",
    exe_name=exe_name,
    model_ws=str(modeldir),
    check=False,
    verbose=True,
)
model_obj.change_model_ws(str(run_dir), reset_external=True)


## Write Model Inputs

This cell writes model input files into the tutorial workspace when supported by the loaded model object. Writing the files before running keeps generated outputs separate from the original source model.


In [ ]:
# Write model files into local run workspace
write_method = getattr(model_obj, "write_simulation", None)
if callable(write_method):
    write_method()
else:
    model_obj.write_input()


## Run The Model

This cell launches MODFLOW-USG through FloPy and reports whether the run succeeded. Stop here if the model does not terminate normally.


In [ ]:
# Run model through FloPy
run_method = getattr(model_obj, "run_simulation", None)
if callable(run_method):
    success, buff = run_method()
else:
    success, buff = model_obj.run_model(silent=False, report=True)

print(f"Success: {success}")
if not success:
    print("Model did not terminate normally.")


## Check Expected Outputs

This cell confirms that expected output files were created before students interpret the run.


In [ ]:
# Output checks
list_candidates = sorted(run_dir.glob("*.lst"))
for f in list_candidates[:5]:
    print(f"List file: {f.name}")

if not list_candidates:
    print("No .lst file found yet in run workspace.")
